# leetcode 365

In [11]:
import logging

logging.basicConfig(level="DEBUG")
logger = logging.getLogger()

cache = {}
count = 0
SEEN = -1

def can_measure(x, y, target, xvol=0, yvol=0):
    global cache, count

    logger.debug(f"starting point: {x}[{xvol}], {y}[{yvol}]")
    if count == 50:
        raise ValueError("max calls exceed")

    count += 1
    
    try:
        res = cache[x, y, target, xvol, yvol]
        if res == SEEN:
            logger.warning(f"been here without result: {xvol=}, {yvol=}")
            return False
        logger.debug(f"cache found: {xvol=}, {yvol=} -> {res}")
        return res
    except KeyError:
        logger.debug(f"set cache to seen: {x}[{xvol}], {y}[{yvol}]")
        cache[x, y, target, xvol, yvol] = SEEN

        
    if x + y < target:
        logger.debug(f"{x=} + {y=} < {target=}, no solution")
        cache[x, y, target, xvol, yvol] = False
        return False
    elif xvol + yvol == target:
        logger.debug(f"found solution: {xvol=}, {yvol=}")
        cache[x, y, target, xvol, yvol] = True
        return True

    # There are 6 paths:
    # 1. Fill x completely
    # 2. Fill y completely
    # 3. Empty x
    # 4. Empty y
    # 5. Transfer x -> y
    # 6. Transfer y -> x

    # Path 1: Fill x
    if xvol < x:
        logger.info(f"fill x: {x}[{xvol}]")
        cache[x, y, target, xvol, yvol] = True
        if can_measure(x, y, target, x, yvol):
            return True

    # Path 2: Fill y
    if yvol < y:
        logger.info(f"fill y: {y}[{yvol}]")
        cache[x, y, target, xvol, yvol] = True
        if can_measure(x, y, target, xvol, y):
            return True

    # Path 3: Empty x
    if xvol > 0:
        logger.info(f"empty x: {x}[{xvol}]")
        cache[x, y, target, xvol, yvol] = True
        if can_measure(x, y, target, 0, yvol):
            return True

    # Path 4: Empty y
    if yvol > 0:
        logger.debug(f"empty y: {y}[{yvol}]")
        cache[x, y, target, xvol, yvol] = True
        if can_measure(x, y, target, xvol, 0):
            return True

    # Path 5: Transfer x -> y
    if xvol > 0 and yvol < y:
        transfer_amount = min(xvol, y - yvol)
        logger.info(f"transfer {transfer_amount} x -> y: {x}[{xvol}] -> {y}[{yvol}]")
        if can_measure(x, y, target, x - transfer_amount, yvol + transfer_amount):
            cache[x, y, target, xvol, yvol] = True
            return True

    # Path 6: Transfer y -> x
    if yvol > 0 and xvol < x:
        transfer_amount = min(yvol, x - xvol)
        logger.info(f"transfer {transfer_amount} y -> x: {x}[{xvol}] <- {y}[{yvol}]")
        if can_measure(x, y, target, xvol + transfer_amount, yvol - transfer_amount):
            cache[x, y, target, xvol, yvol] = True
            return True

    cache[x, y, target, xvol, yvol] = False
    return False    

In [9]:
scenarios = [
    # x, y, target, expected
    (5, 2, 7, True),
    (3, 5, 4, True),
    (1, 10, 8, False),
]
for x, y, target, expected in scenarios:
    logger.info("-----------------------------------")
    logger.info(f"# {x=}, {y=}, {target=}, {expected=}")
    if can_measure(x, y, target) is expected:
        logger.info("  Passed")
    else:
        logger.error("  Failed")

INFO:root:-----------------------------------
INFO:root:# x=5, y=2, target=7, expected=True
DEBUG:root:starting point: 5[0], 2[0]
DEBUG:root:cache found: xvol=0, yvol=0 -> True
INFO:root:  Passed
INFO:root:-----------------------------------
INFO:root:# x=3, y=5, target=4, expected=True
DEBUG:root:starting point: 3[0], 5[0]
DEBUG:root:cache found: xvol=0, yvol=0 -> True
INFO:root:  Passed
INFO:root:-----------------------------------
INFO:root:# x=1, y=10, target=8, expected=False
DEBUG:root:starting point: 1[0], 10[0]
DEBUG:root:set cache to seen: 1[0], 10[0]
INFO:root:fill x: 1[0]
DEBUG:root:starting point: 1[1], 10[0]
DEBUG:root:set cache to seen: 1[1], 10[0]
INFO:root:fill y: 10[0]
DEBUG:root:starting point: 1[1], 10[10]
DEBUG:root:set cache to seen: 1[1], 10[10]
INFO:root:empty x: 1[1]
DEBUG:root:starting point: 1[0], 10[10]
DEBUG:root:set cache to seen: 1[0], 10[10]
INFO:root:fill x: 1[0]
DEBUG:root:starting point: 1[1], 10[10]
DEBUG:root:cache found: xvol=1, yvol=10 -> True
ERRO

In [10]:
cache

{(5, 2, 7, True, 0): True,
 (5, 2, 7, 5, 0): True,
 (5, 2, 7, 5, 2): True,
 (5, 2, 7, 0, 0): True,
 (3, 5, 4, 0, 0): True,
 (3, 5, 4, 3, 0): True,
 (3, 5, 4, 3, 5): True,
 (3, 5, 4, 0, 5): True,
 (1, 10, 8, 0, 0): True,
 (1, 10, 8, 1, 0): True,
 (1, 10, 8, 1, 10): True,
 (1, 10, 8, 0, 10): True}